<a href="https://colab.research.google.com/github/G25ait2026/AIcopy/blob/main/trials.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from collections import deque

# ----------------------------
# Configuration
# ----------------------------

GOAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0   # 0 = Blank (B)
)

MOVES = {
    "Up": -3,
    "Down": 3,
    "Left": -1,
    "Right": 1
}

# ----------------------------
# Helper Functions
# ----------------------------

def print_state(state):
    """Pretty print the 3x3 grid"""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else "B" for x in row))
    print()

def find_blank(state):
    return state.index(0)

def is_valid_move(blank, move):
    if move == "Left" and blank % 3 == 0:
        return False
    if move == "Right" and blank % 3 == 2:
        return False
    if move == "Up" and blank < 3:
        return False
    if move == "Down" and blank > 5:
        return False
    return True

def apply_move(state, blank, move):
    new_blank = blank + MOVES[move]
    new_state = list(state)
    new_state[blank], new_state[new_blank] = new_state[new_blank], new_state[blank]
    return tuple(new_state)

def reconstruct_path(parent, goal):
    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]
    return path[::-1]

# ----------------------------
# Breadth-First Search (BFS)
# ----------------------------

def bfs(initial_state):
    queue = deque([initial_state])
    visited = set([initial_state])
    parent = {initial_state: None}

    while queue:
        current = queue.popleft()

        if current == GOAL_STATE:
            return reconstruct_path(parent, current)

        blank = find_blank(current)

        for move in MOVES:
            if is_valid_move(blank, move):
                next_state = apply_move(current, blank, move)

                if next_state not in visited:
                    visited.add(next_state)
                    parent[next_state] = current
                    queue.append(next_state)

    return None

# ----------------------------
# Main Program
# ----------------------------

if __name__ == "__main__":

    initial_state = (
        1, 2, 3,
        4, 0, 6,
        7, 5, 8
    )

    print("Initial State:")
    print_state(initial_state)

    solution = bfs(initial_state)

    if solution is None:
        print("No solution found.")
    else:
        print("Solution Path:\n")
        for step, state in enumerate(solution):
            print(f"Step {step}:")
            print_state(state)

        print(f"✅ Total moves required: {len(solution) - 1}")


Initial State:
1 2 3
4 B 6
7 5 8

Solution Path:

Step 0:
1 2 3
4 B 6
7 5 8

Step 1:
1 2 3
4 5 6
7 B 8

Step 2:
1 2 3
4 5 6
7 8 B

✅ Total moves required: 2


In [14]:
import heapq
import itertools


class ManuscriptPuzzle:
    def __init__(self, start_state, goal_state):
        self.start_state = tuple(start_state)
        self.goal_state = tuple(goal_state)
        # Pre-calculate goal positions for faster heuristic calculation
        self.goal_positions = {val: (i // 3, i % 3) for i, val in enumerate(self.goal_state)}


    def is_solvable(self):
        """Checks the parity of inversions to see if the puzzle can be solved."""
        flat_list = [tile for tile in self.start_state if tile != 'B']
        inversions = 0
        for i in range(len(flat_list)):
            for j in range(i + 1, len(flat_list)):
                if flat_list[i] > flat_list[j]:
                    inversions += 1
        return inversions % 2 == 0


    def get_manhattan_distance(self, state):
        """Heuristic function: sum of vertical and horizontal distances."""
        distance = 0
        for i, val in enumerate(state):
            if val != 'B':
                curr_row, curr_col = i // 3, i % 3
                goal_row, goal_col = self.goal_positions[val]
                distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
        return distance


    def get_neighbors(self, state):
        """Generates valid moves (Up, Down, Left, Right)."""
        neighbors = []
        state_list = list(state)
        b_idx = state_list.index('B')
        r, c = b_idx // 3, b_idx % 3

        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                neighbor_list = state_list[:]
                target_idx = nr * 3 + nc
                # Swap blank with the manuscript
                neighbor_list[b_idx], neighbor_list[target_idx] = neighbor_list[target_idx], neighbor_list[b_idx]
                neighbors.append(tuple(neighbor_list))
        return neighbors


    def solve(self):
        """Solves the puzzle using A* Search."""
        if not self.is_solvable():
            return "IMPOSSIBLE"


        counter = itertools.count() # Tie-breaker to prevent TypeError
        # Priority Queue: (f_score, tie_breaker, g_score, current_state, path)
        open_set = [(self.get_manhattan_distance(self.start_state), next(counter), 0, self.start_state, [])]
        visited = {self.start_state: 0}


        while open_set:
            f, _, g, current, path = heapq.heappop(open_set)


            if current == self.goal_state:
                return path + [current]


            for neighbor in self.get_neighbors(current):
                new_g = g + 1
                if neighbor not in visited or new_g < visited[neighbor]:
                    visited[neighbor] = new_g
                    h = self.get_manhattan_distance(neighbor)
                    heapq.heappush(open_set, (new_g + h, next(counter), new_g, neighbor, path + [current]))
        return None


# --- EXECUTION ---
# Define the Goal State as specified in your problem: 1 2 3 / 4 5 6 / 7 8 B
goal = [1, 2, 3, 4, 5, 6, 7, 8, 'B']


# Example Scrambled Start (This one is solvable)
start = [1, 2, 3, 4, 8, 5, 7, 6, 'B']


puzzle = ManuscriptPuzzle(start, goal)
solution = puzzle.solve()


if solution == "IMPOSSIBLE":
    print("This specific configuration cannot be solved due to its inversion parity.")
elif solution:
    print(f"Solved in {len(solution) - 1} steps:")
    for i, step in enumerate(solution):
        print(f"Step {i}:")
        print(step[0:3])
        print(step[3:6])
        print(step[6:9])
        print("-" * 10)
else:
    print("No solution found.")

Solved in 4 steps:
Step 0:
(1, 2, 3)
(4, 8, 5)
(7, 6, 'B')
----------
Step 1:
(1, 2, 3)
(4, 8, 5)
(7, 'B', 6)
----------
Step 2:
(1, 2, 3)
(4, 'B', 5)
(7, 8, 6)
----------
Step 3:
(1, 2, 3)
(4, 5, 'B')
(7, 8, 6)
----------
Step 4:
(1, 2, 3)
(4, 5, 6)
(7, 8, 'B')
----------


In [8]:
# ----------------------------
# Depth-First Search (DFS)
# ----------------------------

GOAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0   # 0 = Blank (B)
)

MOVES = {
    "Up": -3,
    "Down": 3,
    "Left": -1,
    "Right": 1
}

def print_state(state):
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else "B" for x in row))
    print()

def find_blank(state):
    return state.index(0)

def is_valid_move(blank, move):
    if move == "Left" and blank % 3 == 0:
        return False
    if move == "Right" and blank % 3 == 2:
        return False
    if move == "Up" and blank < 3:
        return False
    if move == "Down" and blank > 5:
        return False
    return True

def apply_move(state, blank, move):
    new_blank = blank + MOVES[move]
    new_state = list(state)
    new_state[blank], new_state[new_blank] = new_state[new_blank], new_state[blank]
    return tuple(new_state)

def reconstruct_path(parent, goal):
    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]
    return path[::-1]

# ----------------------------
# DFS Algorithm
# ----------------------------

def dfs(initial_state):
    stack = [initial_state]
    visited = set([initial_state])
    parent = {initial_state: None}

    while stack:
        current = stack.pop()

        if current == GOAL_STATE:
            return reconstruct_path(parent, current)

        blank = find_blank(current)

        # Explore deep by pushing neighbors onto stack
        for move in MOVES:
            if is_valid_move(blank, move):
                next_state = apply_move(current, blank, move)

                if next_state not in visited:
                    visited.add(next_state)
                    parent[next_state] = current
                    stack.append(next_state)

    return None

# ----------------------------
# Main Program
# ----------------------------

if __name__ == "__main__":

    initial_state = (
        1, 2, 3,
        4, 0, 6,
        7, 5, 8
    )

    print("Initial State:")
    print_state(initial_state)

    solution = dfs(initial_state)

    if solution is None:
        print("No solution found.")
    else:
        print("DFS Solution Path:\n")
        for step, state in enumerate(solution):
            print(f"Step {step}:")
            print_state(state)

        print(f"⚠️ Total moves found by DFS: {len(solution) - 1}")


Streaming output truncated to the last 5000 lines.
8 3 5
B 1 7
6 4 2

Step 48286:
8 3 5
1 B 7
6 4 2

Step 48287:
8 3 5
1 7 B
6 4 2

Step 48288:
8 3 B
1 7 5
6 4 2

Step 48289:
8 B 3
1 7 5
6 4 2

Step 48290:
8 7 3
1 B 5
6 4 2

Step 48291:
8 7 3
1 5 B
6 4 2

Step 48292:
8 7 B
1 5 3
6 4 2

Step 48293:
8 B 7
1 5 3
6 4 2

Step 48294:
B 8 7
1 5 3
6 4 2

Step 48295:
1 8 7
B 5 3
6 4 2

Step 48296:
1 8 7
6 5 3
B 4 2

Step 48297:
1 8 7
6 5 3
4 B 2

Step 48298:
1 8 7
6 5 3
4 2 B

Step 48299:
1 8 7
6 5 B
4 2 3

Step 48300:
1 8 7
6 B 5
4 2 3

Step 48301:
1 8 7
B 6 5
4 2 3

Step 48302:
B 8 7
1 6 5
4 2 3

Step 48303:
8 B 7
1 6 5
4 2 3

Step 48304:
8 7 B
1 6 5
4 2 3

Step 48305:
8 7 5
1 6 B
4 2 3

Step 48306:
8 7 5
1 6 3
4 2 B

Step 48307:
8 7 5
1 6 3
4 B 2

Step 48308:
8 7 5
1 6 3
B 4 2

Step 48309:
8 7 5
B 6 3
1 4 2

Step 48310:
B 7 5
8 6 3
1 4 2

Step 48311:
7 B 5
8 6 3
1 4 2

Step 48312:
7 6 5
8 B 3
1 4 2

Step 48313:
7 6 5
8 3 B
1 4 2

Step 48314:
7 6 B
8 3 5
1 4 2

Step 48315:
7 B 6
8 3 5
1 4 2



In [9]:
import heapq

# ----------------------------
# Configuration
# ----------------------------

GOAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0   # 0 = Blank (B)
)

MOVES = {
    "Up": -3,
    "Down": 3,
    "Left": -1,
    "Right": 1
}

# ----------------------------
# Helper Functions
# ----------------------------

def print_state(state):
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else "B" for x in row))
    print()

def find_blank(state):
    return state.index(0)

def is_valid_move(blank, move):
    if move == "Left" and blank % 3 == 0:
        return False
    if move == "Right" and blank % 3 == 2:
        return False
    if move == "Up" and blank < 3:
        return False
    if move == "Down" and blank > 5:
        return False
    return True

def apply_move(state, blank, move):
    new_blank = blank + MOVES[move]
    new_state = list(state)
    new_state[blank], new_state[new_blank] = new_state[new_blank], new_state[blank]
    return tuple(new_state)

def reconstruct_path(parent, goal):
    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]
    return path[::-1]

# ----------------------------
# Heuristic: Manhattan Distance
# ----------------------------

def manhattan_distance(state):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            goal_index = GOAL_STATE.index(state[i])
            x1, y1 = divmod(i, 3)
            x2, y2 = divmod(goal_index, 3)
            distance += abs(x1 - x2) + abs(y1 - y2)
    return distance

# ----------------------------
# Greedy Best-First Search
# ----------------------------

def greedy_best_first_search(initial_state):
    priority_queue = []
    heapq.heappush(priority_queue, (manhattan_distance(initial_state), initial_state))

    visited = set([initial_state])
    parent = {initial_state: None}

    while priority_queue:
        _, current = heapq.heappop(priority_queue)

        if current == GOAL_STATE:
            return reconstruct_path(parent, current)

        blank = find_blank(current)

        for move in MOVES:
            if is_valid_move(blank, move):
                next_state = apply_move(current, blank, move)

                if next_state not in visited:
                    visited.add(next_state)
                    parent[next_state] = current
                    h = manhattan_distance(next_state)
                    heapq.heappush(priority_queue, (h, next_state))

    return None

# ----------------------------
# Main Program
# ----------------------------

if __name__ == "__main__":

    initial_state = (
        1, 2, 3,
        4, 0, 6,
        7, 5, 8
    )

    print("Initial State:")
    print_state(initial_state)

    solution = greedy_best_first_search(initial_state)

    if solution is None:
        print("No solution found.")
    else:
        print("Greedy Best-First Search Solution:\n")
        for step, state in enumerate(solution):
            print(f"Step {step}:")
            print_state(state)

        print(f"⚡ Total moves (Greedy): {len(solution) - 1}")


Initial State:
1 2 3
4 B 6
7 5 8

Greedy Best-First Search Solution:

Step 0:
1 2 3
4 B 6
7 5 8

Step 1:
1 2 3
4 5 6
7 B 8

Step 2:
1 2 3
4 5 6
7 8 B

⚡ Total moves (Greedy): 2


In [10]:
import heapq

# ----------------------------
# Configuration
# ----------------------------

GOAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0   # 0 = Blank (B)
)

MOVES = {
    "Up": -3,
    "Down": 3,
    "Left": -1,
    "Right": 1
}

# ----------------------------
# Helper Functions
# ----------------------------

def print_state(state):
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else "B" for x in row))
    print()

def find_blank(state):
    return state.index(0)

def is_valid_move(blank, move):
    if move == "Left" and blank % 3 == 0:
        return False
    if move == "Right" and blank % 3 == 2:
        return False
    if move == "Up" and blank < 3:
        return False
    if move == "Down" and blank > 5:
        return False
    return True

def apply_move(state, blank, move):
    new_blank = blank + MOVES[move]
    new_state = list(state)
    new_state[blank], new_state[new_blank] = new_state[new_blank], new_state[blank]
    return tuple(new_state)

def reconstruct_path(parent, goal):
    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]
    return path[::-1]

# ----------------------------
# Heuristics
# ----------------------------

def misplaced_tiles(state):
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != GOAL_STATE[i]:
            count += 1
    return count

def manhattan_distance(state):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            goal_index = GOAL_STATE.index(state[i])
            x1, y1 = divmod(i, 3)
            x2, y2 = divmod(goal_index, 3)
            distance += abs(x1 - x2) + abs(y1 - y2)
    return distance

# ----------------------------
# A* Search Algorithm
# ----------------------------

def a_star_search(initial_state, heuristic):
    priority_queue = []
    heapq.heappush(priority_queue, (0, initial_state))

    parent = {initial_state: None}
    g_cost = {initial_state: 0}
    visited = set()

    while priority_queue:
        _, current = heapq.heappop(priority_queue)

        if current == GOAL_STATE:
            return reconstruct_path(parent, current)

        visited.add(current)
        blank = find_blank(current)

        for move in MOVES:
            if is_valid_move(blank, move):
                next_state = apply_move(current, blank, move)
                tentative_g = g_cost[current] + 1

                if next_state in visited:
                    continue

                if next_state not in g_cost or tentative_g < g_cost[next_state]:
                    g_cost[next_state] = tentative_g
                    f_cost = tentative_g + heuristic(next_state)
                    heapq.heappush(priority_queue, (f_cost, next_state))
                    parent[next_state] = current

    return None

# ----------------------------
# Main Program
# ----------------------------

if __name__ == "__main__":

    initial_state = (
        1, 2, 3,
        4, 0, 6,
        7, 5, 8
    )

    print("Initial State:")
    print_state(initial_state)

    print("A* using Misplaced Tiles Heuristic (h1):\n")
    solution_h1 = a_star_search(initial_state, misplaced_tiles)

    for step, state in enumerate(solution_h1):
        print(f"Step {step}:")
        print_state(state)

    print(f"Total moves (h1): {len(solution_h1) - 1}\n")

    print("A* using Manhattan Distance Heuristic (h2):\n")
    solution_h2 = a_star_search(initial_state, manhattan_distance)

    for step, state in enumerate(solution_h2):
        print(f"Step {step}:")
        print_state(state)

    print(f"Total moves (h2): {len(solution_h2) - 1}")


Initial State:
1 2 3
4 B 6
7 5 8

A* using Misplaced Tiles Heuristic (h1):

Step 0:
1 2 3
4 B 6
7 5 8

Step 1:
1 2 3
4 5 6
7 B 8

Step 2:
1 2 3
4 5 6
7 8 B

Total moves (h1): 2

A* using Manhattan Distance Heuristic (h2):

Step 0:
1 2 3
4 B 6
7 5 8

Step 1:
1 2 3
4 5 6
7 B 8

Step 2:
1 2 3
4 5 6
7 8 B

Total moves (h2): 2
